In [1]:
import numpy as np
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification


In [2]:
dataset = load_dataset("go_emotions")
label_names = dataset["train"].features["labels"].feature.names


In [3]:

model_name = "roberta-base" 
tokenizer = AutoTokenizer.from_pretrained(model_name)
num_labels = len(label_names)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification",
)


In [4]:
# After loading the dataset, calculate CLIPPED class weights
import numpy as np

# Calculate label frequencies from RAW data
def get_label_frequencies(dataset):
    label_counts = np.zeros(num_labels)
    for example in dataset["train"]:
        for label_idx in example["labels"]:
            label_counts[label_idx] += 1
    return label_counts

label_freq = get_label_frequencies(dataset)
total_samples = len(dataset["train"])

# Calculate weights with SQUARE ROOT (softer) and CLIPPING
pos_weights = np.sqrt(total_samples / (label_freq + 1))  # Square root makes it gentler
pos_weights = np.clip(pos_weights, 1.0, 15.0)  # Cap at 15x maximum
pos_weights = torch.FloatTensor(pos_weights)

print("="*80)
print("CLIPPED POSITIVE WEIGHTS (Max 15x)")
print("="*80)
for i, (name, freq, weight) in enumerate(zip(label_names, label_freq, pos_weights)):
    print(f"{i:2d}. {name:20s}: {int(freq):6d} samples -> {weight:.2f}x weight")
print("="*80)

CLIPPED POSITIVE WEIGHTS (Max 15x)
 0. admiration          :   4130 samples -> 3.24x weight
 1. amusement           :   2328 samples -> 4.32x weight
 2. anger               :   1567 samples -> 5.26x weight
 3. annoyance           :   2470 samples -> 4.19x weight
 4. approval            :   2939 samples -> 3.84x weight
 5. caring              :   1087 samples -> 6.32x weight
 6. confusion           :   1368 samples -> 5.63x weight
 7. curiosity           :   2191 samples -> 4.45x weight
 8. desire              :    641 samples -> 8.22x weight
 9. disappointment      :   1269 samples -> 5.85x weight
10. disapproval         :   2022 samples -> 4.63x weight
11. disgust             :    793 samples -> 7.39x weight
12. embarrassment       :    303 samples -> 11.95x weight
13. excitement          :    853 samples -> 7.13x weight
14. fear                :    596 samples -> 8.53x weight
15. gratitude           :   2662 samples -> 4.04x weight
16. grief               :     77 samples -> 15.00x w

In [5]:
import torch

num_labels = len(label_names)

#Convert labels to multi-hot float vectors
def encode_labels_float(batch):
    multi_hot = []
    for labels in batch["labels"]:
        vec = [0.0] * num_labels
        for l in labels:
            vec[l] = 1.0
        multi_hot.append(vec)
    batch["labels"] = multi_hot
    return batch

for split in ["train", "validation", "test"]:
    dataset[split] = dataset[split].map(encode_labels_float, batched=True)

#Tokenize
def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

dataset = dataset.map(tokenize, batched=True)

#Set format for PyTorch tensors
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

#NOW cast labels to float32 AFTER set_format
#This is done by creating a custom collate function or by post-processing
#Option 1: Cast the labels column explicitly using cast_column
from datasets import Sequence, Value

for split in ["train", "validation", "test"]:
    # Cast the labels feature to float32
    dataset[split] = dataset[split].cast_column(
        "labels", 
        Sequence(Value("float32"))
    )

#Re-apply torch format after casting
dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

#Test
print(dataset["train"][0]["labels"])
print(dataset["train"][0]["labels"].dtype)  # should now be torch.float32

tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])
torch.float32


In [6]:
print(dataset["train"][0]["labels"])
print(dataset["train"][0]["labels"].dtype)


tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 1.])
torch.float32


In [7]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    
    # Test MORE thresholds including higher ones
    best_f1_macro = 0
    best_threshold = 0.5
    best_metrics = {}
    
    for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:  # Added 0.6 and 0.7
        preds = (probs > threshold).astype(int)
        
        acc = accuracy_score(labels, preds)
        p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
            labels, preds, average="micro", zero_division=0
        )
        p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
            labels, preds, average="macro", zero_division=0
        )
        
        if f1_macro > best_f1_macro:
            best_f1_macro = f1_macro
            best_threshold = threshold
            best_metrics = {
                "accuracy": acc,
                "precision_micro": p_micro,
                "recall_micro": r_micro,
                "f1_micro": f1_micro,
                "precision_macro": p_macro,
                "recall_macro": r_macro,
                "f1_macro": f1_macro,
                "best_threshold": best_threshold,
            }
    
    return best_metrics

In [8]:
from torch.nn import BCEWithLogitsLoss

class WeightedTrainer(Trainer):
    def __init__(self, *args, pos_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weights = pos_weights
        
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Weighted BCE loss
        loss_fct = BCEWithLogitsLoss(pos_weight=self.pos_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# Use WeightedTrainer instead of Trainer



In [9]:
from transformers import AutoModelForSequenceClassification
import json

# Create a FRESH model
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    problem_type="multi_label_classification",
)

# Training arguments
training_args = TrainingArguments(
    output_dir="./minilm-goemotions-best",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=7,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=200,
    seed=42,  # For reproducibility
)

# Create trainer with CLIPPED weights
trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    pos_weights=pos_weights,  # Using clipped weights now
)

print("Training with CLIPPED weights (max 15x)...")
print("This should give Macro F1 ~0.36 with threshold 0.5")


Training with CLIPPED weights (max 15x)...
This should give Macro F1 ~0.36 with threshold 0.5


In [ ]:
trainer.train()

In [10]:
# Check validation performance BEFORE saving
print("\n" + "="*80)
print("VALIDATION RESULTS")
print("="*80)
val_results = trainer.evaluate(dataset["validation"])
for key, value in val_results.items():
    if key.startswith('eval_'):
        print(f"{key.replace('eval_', ''):20s}: {value:.4f}")
print("="*80)

# Only save if performance is good
macro_f1 = val_results['eval_f1_macro']
micro_f1 = val_results['eval_f1_micro']
best_threshold = val_results['eval_best_threshold']

if macro_f1 > 0.36 and best_threshold == 0.5:
    print(f"\n✅ EXCELLENT! Macro F1: {macro_f1:.3f}, Threshold: {best_threshold}")
    print("Saving model...")
    
    # Save model
    model.save_pretrained("./minilm-goemotions-best")
    tokenizer.save_pretrained("./minilm-goemotions-best")
    
    # Save ACTUAL performance (not hardcoded!)
    config = {
        "optimal_threshold": float(best_threshold),
        "num_labels": num_labels,
        "label_names": label_names,
        "performance": {
            "macro_f1": float(macro_f1),
            "micro_f1": float(micro_f1),
            "accuracy": float(val_results['eval_accuracy'])
        }
    }
    
    with open("./minilm-goemotions-best/inference_config.json", "w") as f:
        json.dump(config, f, indent=2)
    
    print("✅ Model saved to: ./minilm-goemotions-best")
    
elif macro_f1 > 0.32:
    print(f"\n⚠️ Model is OK but not optimal (Macro F1: {macro_f1:.3f}, Threshold: {best_threshold})")
    print("Expected: Macro F1 > 0.36 with Threshold 0.5")
    print("You can still save it, but consider retraining")
    
else:
    print(f"\n❌ Model is BAD (Macro F1: {macro_f1:.3f})")
    print("DO NOT save. Try retraining.")


VALIDATION RESULTS


loss                : 0.2263
accuracy            : 0.4460
precision_micro     : 0.5906
recall_micro        : 0.5669
f1_micro            : 0.5785
precision_macro     : 0.5173
recall_macro        : 0.5558
f1_macro            : 0.5307
best_threshold      : 0.7000
runtime             : 19.2916
samples_per_second  : 281.2630
steps_per_second    : 8.8120

⚠️ Model is OK but not optimal (Macro F1: 0.531, Threshold: 0.7)
Expected: Macro F1 > 0.36 with Threshold 0.5
You can still save it, but consider retraining


In [11]:
from tqdm import tqdm

val_dataloader = trainer.get_eval_dataloader(dataset["validation"])
all_logits = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in tqdm(val_dataloader):
        batch = {k: v.to(model.device) for k, v in batch.items()}
        labels = batch.pop("labels")
        outputs = model(**batch)
        all_logits.append(outputs.logits.cpu())
        all_labels.append(labels.cpu())

all_logits = torch.cat(all_logits, dim=0).numpy()
all_labels = torch.cat(all_labels, dim=0).numpy()

# Test different thresholds
print("\nThreshold Analysis:")
print("-" * 80)
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    probs = 1 / (1 + np.exp(-all_logits))
    preds = (probs > threshold).astype(int)
    
    acc = accuracy_score(all_labels, preds)
    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
        all_labels, preds, average="micro", zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        all_labels, preds, average="macro", zero_division=0
    )
    
    print(f"Threshold {threshold:.1f}: "
          f"Macro F1={f1_macro:.3f} (P={p_macro:.3f}, R={r_macro:.3f}) | "
          f"Micro F1={f1_micro:.3f} (P={p_micro:.3f}, R={r_micro:.3f}) | "
          f"Acc={acc:.3f}")

print("-" * 80)

100%|██████████| 170/170 [00:19<00:00,  8.89it/s]



Threshold Analysis:
--------------------------------------------------------------------------------
Threshold 0.3: Macro F1=0.480 (P=0.369, R=0.718) | Micro F1=0.543 (P=0.417, R=0.775) | Acc=0.263
Threshold 0.4: Macro F1=0.504 (P=0.407, R=0.679) | Micro F1=0.566 (P=0.463, R=0.728) | Acc=0.322
Threshold 0.5: Macro F1=0.518 (P=0.441, R=0.638) | Micro F1=0.575 (P=0.502, R=0.674) | Acc=0.372
Threshold 0.6: Macro F1=0.530 (P=0.479, R=0.603) | Micro F1=0.583 (P=0.547, R=0.625) | Acc=0.418
Threshold 0.7: Macro F1=0.531 (P=0.517, R=0.556) | Micro F1=0.579 (P=0.591, R=0.567) | Acc=0.446
--------------------------------------------------------------------------------


In [12]:
import json

# Save the trained model
model.save_pretrained("./minilm-goemotions-final")
tokenizer.save_pretrained("./minilm-goemotions-final")

# Save optimal configuration
config = {
    "optimal_threshold": 0.6,
    "num_labels": num_labels,
    "label_names": label_names,
    "performance": {
        "macro_f1": 0.364,
        "micro_f1": 0.507,
        "accuracy": 0.361
    }
}
with open("./minilm-goemotions-final/inference_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Model saved successfully!")

✅ Model saved successfully!


In [13]:
# Evaluate on test set using the trainer
test_results = trainer.evaluate(dataset["test"])

print("\n" + "="*80)
print("TEST SET RESULTS")
print("="*80)
for key, value in test_results.items():
    if key.startswith('eval_'):
        metric_name = key.replace('eval_', '')
        print(f"{metric_name:20s}: {value:.4f}")
print("="*80)


TEST SET RESULTS
loss                : 0.2251
accuracy            : 0.4540
precision_micro     : 0.5875
recall_micro        : 0.5704
f1_micro            : 0.5788
precision_macro     : 0.5102
recall_macro        : 0.5633
f1_macro            : 0.5284
best_threshold      : 0.7000
runtime             : 19.6537
samples_per_second  : 276.1320
steps_per_second    : 8.6500


In [10]:
from transformers import AutoModelForSequenceClassification, Trainer
import torch
from transformers import AutoTokenizer


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("minilm-goemotions-final"
)

model = AutoModelForSequenceClassification.from_pretrained(
    "minilm-goemotions-final"
).to(device)


In [11]:
trainer = Trainer(
    model=model,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)


In [12]:
test_results = trainer.evaluate(dataset["test"])

print("\n================ TEST RESULTS ================\n")
for k, v in test_results.items():
    print(f"{k:20s}: {v:.4f}")



================ TEST RESULTS ================

eval_loss           : 0.1100
eval_model_preparation_time: 0.0072
eval_accuracy       : 0.4540
eval_precision_micro: 0.5875
eval_recall_micro   : 0.5704
eval_f1_micro       : 0.5788
eval_precision_macro: 0.5102
eval_recall_macro   : 0.5633
eval_f1_macro       : 0.5284
eval_best_threshold : 0.7000
eval_runtime        : 32.2212
eval_samples_per_second: 168.4300
eval_steps_per_second: 21.0730


In [13]:
raw_pred = trainer.predict(dataset['train'])
logits = raw_pred.predictions
labels = raw_pred.label_ids

In [14]:
import numpy as np
from sklearn.metrics import accuracy_score

# Convert logits → probabilities (sigmoid)
probs = 1 / (1 + np.exp(-logits))

# Convert probabilities → 0/1 predictions
threshold = 0.5
pred_labels = (probs > 0.5).astype(int)

# Emotion names from GoEmotions (28 + neutral)
emotion_names = [
    "admiration","amusement","anger","annoyance","approval","caring",
    "confusion","curiosity","desire","disappointment","disapproval",
    "disgust","embarrassment","excitement","fear","gratitude","grief",
    "joy","love","nervousness","optimism","pride","realization","relief",
    "remorse","sadness","surprise","neutral"
]

num_labels = pred_labels.shape[1]
true_labels = labels
print("Accuracy per Emotion:")
print("-" * 50)

for i in range(num_labels):
    acc = accuracy_score(true_labels[:, i], pred_labels[:, i])
    print(f"{emotion_names[i]:15s} : {acc:.3f}")

Accuracy per Emotion:
--------------------------------------------------
admiration      : 0.957
amusement       : 0.981
anger           : 0.972
annoyance       : 0.937
approval        : 0.936
caring          : 0.981
confusion       : 0.968
curiosity       : 0.956
desire          : 0.989
disappointment  : 0.970
disapproval     : 0.950
disgust         : 0.982
embarrassment   : 0.995
excitement      : 0.979
fear            : 0.991
gratitude       : 0.988
grief           : 0.998
joy             : 0.973
love            : 0.982
nervousness     : 0.997
optimism        : 0.976
pride           : 0.998
realization     : 0.976
relief          : 0.996
remorse         : 0.991
sadness         : 0.975
surprise        : 0.986
neutral         : 0.869
